## Create function that loads in one 8-day average of MODIS data and plots chla in our study region

In [1]:
import numpy as np
from oneargopy.OneArgo import Argo
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import matplotlib.cm as cm # new package for the colorbar
import matplotlib.patches as mpatches
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.ticker import LogLocator, FormatStrFormatter
from matplotlib.colors import LogNorm
import gsw
import matplotlib.image as mpimg
import earthaccess
import h5netcdf
import seaborn as sns
import xarray as xr
from matplotlib.patches import Rectangle

/opt/anaconda3/envs/IBIS_Project/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Reading in .csv and adding season/datetime

In [4]:
df_new = pd.read_csv('/Users/lilah/Documents/IBIS_Project/Data_Summer/new_bgc_profiles_BR_extended_area_all_time.csv')

/var/folders/sr/nn_fjjln6s33md98tyfc3rd40000gn/T/ipykernel_32468/3120791858.py:1: DtypeWarning: Columns (0: LATITUDE, 1: LONGITUDE) have mixed types. Specify dtype option on import or set low_memory=False.
  df_new = pd.read_csv('/Users/lilah/Documents/IBIS_Project/Data_Summer/new_bgc_profiles_BR_extended_area_all_time.csv')


In [5]:
# NEW DATA

#convert to datetime
df_new['DATE'] = pd.to_datetime(df_new['DATE'],format = 'mixed')
df_new = df_new.dropna(subset=["DATE"]) #drop any cycles with no date or lat/lon (I found one cycle on a float that was causing these errors)

#fix lat lon values with '--' (replace with nan) (no longer needed since I found the problem float)
df_new['LATITUDE'] = pd.to_numeric(df_new['LATITUDE'], errors='coerce')
df_new['LONGITUDE'] = pd.to_numeric(df_new['LONGITUDE'], errors='coerce')


In [6]:
#create season columns
df_new["year"] = df_new["DATE"].dt.year.astype("Int64")
df_new["month"] = df_new["DATE"].dt.month.astype("Int64")

def assign_season(row):
    m = int(row["month"])
    y = int(row["year"])
    if m >= 9:
        return f"{y}-{y+1}"
    elif m <= 4:
        return f"{y-1}-{y}"
    else:
        return f"{y}"

df_new["season"] = df_new.apply(assign_season, axis=1)

## Login to earthaccess

In [2]:
auth = earthaccess.login()

## Search for MODIS data from 2014-2015 in study region

In [3]:
results = earthaccess.search_datasets(instrument="MODIS")

In [4]:
for item in results:
    summary = item.summary()
    print(summary["short-name"])

CIESIN_SEDAC_USPAT_USUEXT2015
Sat_ActiveLayer_Thickness_Maps_1760
ABoVE_MODIS_MAIAC_Reflectance_1858
Wildfires_NWT_Canada_1548
Wildfires_2014_NWT_Canada_1307
Burned_Area_Depth_AK_CA_2063
MODIS_MAIAC_Reflectance_1700
Dall_Sheep_Population_Dynamics_1640
Dall_Sheep_Snowpack_1602
Effect_Environment_Moose_1739
ABoVE_Frac_Open_Water_1362
Snow_Cover_Extent_and_Depth_1757
BurnedArea_Emissions_AK_YT_NWT_1812
Last_Day_Spring_Snow_1528
GPP_MODIS_Alaska_Canada_2024
MODIS_CCaN_NDVI_Trends_Alaska_1666
Albedo_Boreal_North_America_1605
Alaska_Yukon_NDVI_1614
NorthSlope_NEE_TVPRM_1920
Wildfires_Date_of_Burning_1559
Wolves_Denning_Pups_Climate_1846
AGB_Pantropics_Amazon_Mexico_1824
ACTIVATE-MODIS-MERRA2
ADAM.Surface.Reflectance.Database
CIESIN_AfSIS_MODIS_ALB2012
CIESIN_AfSIS_MODIS_LCT2012
CIESIN_AfSIS_MODIS_LST201404
CIESIN_AfSIS_MODIS_LAIFPAR2012
CIESIN_AfSIS_MODIS_PP2012
CIESIN_AfSIS_MODIS_VEGIN201404
aad_ais_gz_modis_slope_break
AMDBLWV
AMDBLWV
AMMBLWV
AMMBLWV
NSIDC-0432
UNIZH_AGREG
CIESIN_SEDAC_SDE

/var/folders/sr/nn_fjjln6s33md98tyfc3rd40000gn/T/ipykernel_32721/2661939172.py:2: FutureWarning: As of version 1.0, `DataCollection.summary` will be accessed as an attribute; e.g. use `DataCollection.summary` **not** `DataCollection.summary()`
  summary = item.summary()
/var/folders/sr/nn_fjjln6s33md98tyfc3rd40000gn/T/ipykernel_32721/2661939172.py:2: FutureWarning: As of version 1.0, `DataCollection.summary` will be accessed as an attribute; e.g. use `DataCollection.summary` **not** `DataCollection.summary()`
  summary = item.summary()
/var/folders/sr/nn_fjjln6s33md98tyfc3rd40000gn/T/ipykernel_32721/2661939172.py:2: FutureWarning: As of version 1.0, `DataCollection.summary` will be accessed as an attribute; e.g. use `DataCollection.summary` **not** `DataCollection.summary()`
  summary = item.summary()
/var/folders/sr/nn_fjjln6s33md98tyfc3rd40000gn/T/ipykernel_32721/2661939172.py:2: FutureWarning: As of version 1.0, `DataCollection.summary` will be accessed as an attribute; e.g. use `Da

In [12]:
tspan = ("2015-01-01", "2015-01-01")
bbox = (150, -78, 160, -70)
# bbox = (155, -65, 160, -60) # west, south, east, north 
clouds = (0, 100)
# cloud_cover=clouds

In [13]:
results = earthaccess.search_data(
    short_name="MODISA_L3m_CHL",
    temporal=tspan,
    bounding_box=bbox,
    granule_name="*.8D.*4*",  # 8-day for MODIS | Resolution: 0p1deg or 4 (for 4km) (want 4 for 4km)
)
len(results)

/opt/anaconda3/envs/IBIS_Project/lib/python3.13/site-packages/earthaccess/results.py:348: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


1

In [10]:
results

/opt/anaconda3/envs/IBIS_Project/lib/python3.13/site-packages/earthaccess/results.py:375: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  Size(MB): {self.size()}


[Collection: {'ShortName': 'MODISA_L3m_CHL', 'Version': '2022.0'}
 Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'BoundingRectangles': [{'SouthBoundingCoordinate': -90, 'EastBoundingCoordinate': 180, 'NorthBoundingCoordinate': 90, 'WestBoundingCoordinate': -180}]}}}
 Temporal coverage: {'RangeDateTime': {'BeginningDateTime': '2014-01-01T00:00:00Z', 'EndingDateTime': '2014-01-08T23:59:59Z'}}
 Size(MB): 42.982041358947754
 Data: ['https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/AQUA_MODIS.20140101_20140108.L3m.8D.CHL.chlor_a.4km.nc'],
 Collection: {'ShortName': 'MODISA_L3m_CHL', 'Version': '2022.0'}
 Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'BoundingRectangles': [{'EastBoundingCoordinate': 180, 'WestBoundingCoordinate': -180, 'NorthBoundingCoordinate': 90, 'SouthBoundingCoordinate': -90}]}}}
 Temporal coverage: {'RangeDateTime': {'EndingDateTime': '2014-01-16T23:59:59Z', 'BeginningDateTime': '2014-01-09T00:00:00Z'}}
 Size(MB): 43.0111465

## Opening files without downloading them (streaming them)

In [11]:
paths = earthaccess.open(results)

/opt/anaconda3/envs/IBIS_Project/lib/python3.13/site-packages/earthaccess/store.py:523: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum([granule.size() for granule in granules]) / 1024, 2)
QUEUEING TASKS | : 100%|██████████| 50/50 [00:00<00:00, 10881.30it/s]
PROCESSING TASKS | : 100%|██████████| 50/50 [00:08<00:00,  5.77it/s]
COLLECTING RESULTS | : 100%|██████████| 50/50 [00:00<00:00, 420270.94it/s]


In [12]:
paths

[<File-like object HTTPFileSystem, https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/AQUA_MODIS.20140101_20140108.L3m.8D.CHL.chlor_a.4km.nc>,
 <File-like object HTTPFileSystem, https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/AQUA_MODIS.20140109_20140116.L3m.8D.CHL.chlor_a.4km.nc>,
 <File-like object HTTPFileSystem, https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/AQUA_MODIS.20140117_20140124.L3m.8D.CHL.chlor_a.4km.nc>,
 <File-like object HTTPFileSystem, https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/AQUA_MODIS.20140125_20140201.L3m.8D.CHL.chlor_a.4km.nc>,
 <File-like object HTTPFileSystem, https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/AQUA_MODIS.20140202_20140209.L3m.8D.CHL.chlor_a.4km.nc>,
 <File-like object HTTPFileSystem, https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/AQUA_MODIS.20140210_20140217.L3m.8D.CHL.chlor_a.4km.nc>,
 <File-like object HTTPFileSystem, https://obdaac-te

## Function

In [3]:
# def sat_data(path, instrument, tspan, bbox, clouds, short_name, granule_name):

results = earthaccess.search_datasets(instrument="MODIS")

# spatio-temporal parameters
tspan = ("2015-01-01", "2015-01-01")
bbox = (150, -78, 160, -70) # west, south, east, north 
clouds = (0, 100)


# getting data files
results = earthaccess.search_data(
short_name="MODISA_L3m_CHL",
temporal=tspan,
bounding_box=bbox,
granule_name="*.8D.*4*")  # 8-day for MODIS | Resolution: 0p1deg or 4 (for 4km) (want 4 for 4km)


# open files without downloading them
paths = earthaccess.open(results)

dataset = xr.open_dataset(paths[0])
datatree = xr.open_datatree(paths[0])
dataset = xr.merge(datatree.to_dict().values())
    
# sat_data(path=0, instrument="MODIS", tspan=("2015-01-01", "2015-01-01"), bbox=(150, -78, 160, -70), clouds=(0, 100), short_name="MODISA_L3m_CHL", granule_name="*.8D.*4*")

/opt/anaconda3/envs/IBIS_Project/lib/python3.13/site-packages/earthaccess/results.py:348: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()
/opt/anaconda3/envs/IBIS_Project/lib/python3.13/site-packages/earthaccess/store.py:523: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum([granule.size() for granule in granules]) / 1024, 2)
QUEUEING TASKS | : 100%|██████████| 1/1 [00:00<00:00, 468.17it/s]
PROCESSING TASKS | : 100%|██████████| 1/1 [00:01<00:00,  1.84s/it]
COLLECTING RESULTS | : 100%|██████████| 1/1 [00:00<00:00, 10754.63it/s]


## This cell is crashing my kernel everytime I run it in or out of the function

In [ ]:
# Opening files into xarrays and merging variables into one
dataset = xr.open_dataset(paths[0])
datatree = xr.open_datatree(paths[0])
dataset = xr.merge(datatree.to_dict().values())
dataarray = dataset.to_dataarray(dataset)


In [5]:
dataset = xr.open_dataset(paths[0])

In [6]:
datatree = xr.open_datatree(paths[0])

In [7]:
dataset = xr.merge(datatree.to_dict().values())

## The culprit has been found

In [ ]:
dataarray = dataset.to_dataarray(dataset)

In [1]:
def sat_plot(data):

 # PLOTTING DATA ------------------------------------------------
    #assign projection type
    crs0 = ccrs.PlateCarree()
   

    #create figure object
    fig = plt.figure(figsize = (20,8))

    #RECCAP regions panel
    ax = fig.add_subplot(111, projection = crs0)

    #Colormap specification
    color = cmocean.cm.haline

    plot = plt.pcolor(data['lon'],data['lat'],data,
                 vmin = 0,vmax = 6) 

    cbar = fig.colorbar(plot, fraction=0.02, pad=0.02)
    # cbar.ax.tick_params(labelsize=tick_size)
    # cbar.set_label('Chla', size=axis_size)

    gl = ax.gridlines(crs=crs0, draw_labels=True, x_inline=False, y_inline=False,
                          linewidth=1, color='gray', alpha=0.6, linestyle='-')

sat_plot(data=dataarray)

NameError: name 'dataarray' is not defined